#  Kişiselleştirilmiş Asistan — Hayvan Verisiyle Birleştirme

**Amaç:** RAG asistanını, belirli bir hayvanın gerçek verileriyle (verim, ağırlık,
anomali durumu) birleştirmek. Böylece asistan genel bilgi yerine, o hayvana özel
kişiselleştirilmiş cevap verir.

**Örnek:** "TR-002 ineğim neden az süt veriyor?" sorusuna, o ineğin kendi
verilerine (düşük ağırlık, anomali işareti vb.) bakarak yanıt vermek.

**Yol haritası:** Bu, AI geliştirme yol haritasının 1. adımıdır.

In [1]:
!pip install sentence-transformers -q

In [2]:
from sentence_transformers import SentenceTransformer, util
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Açıklama: Metinleri anlam vektörüne çeviren embedding modeli — RAG'in getirme kısmının temeli.

In [3]:
bilgi_tabani = [
    "Buzağıların yem yememesinin yaygın sebepleri geçiş dönemi stresi, sindirim bozuklukları, diş çıkarma ağrısı, soğuk veya kirli su, ani yem değişikliği ve bağırsak parazitleridir. Yem değişikliği kademeli yapılmalı, temiz su sağlanmalı ve ortam sıcak tutulmalıdır. İştahsızlık iki günü geçerse veteriner hekime başvurulmalıdır.",
    "Süt veriminin düşmesinin başlıca nedenleri yetersiz veya kalitesiz yem, su tüketiminin azalması, sıcaklık stresi, meme iltihabı (mastitis), laktasyon döneminin ilerlemesi ve gebeliktir. İlk kontrol edilmesi gerekenler yem kalitesi, yem miktarı ve temiz suya erişimdir. Ani düşüşte meme mastitis açısından kontrol edilmeli, sorun sürerse veteriner hekime danışılmalıdır.",
    "İneklerde topallık genellikle tırnak hastalıkları (çürük, çatlak), sert veya sürekli ıslak zemin, eklem iltihabı, yaralanma ve mineral eksikliğinden kaynaklanır. Topallayan hayvanın yem tüketimi ve süt verimi düşebilir. Erken fark edilirse tırnak bakımı ve zemin düzenlemesiyle tedavi başarısı yüksektir; ilerlemiş durumlarda veteriner müdahalesi gerekir.",
    "Gebe ineklerin doğuma yakın kuru döneminde beslenmesi kritiktir. Enerji ve protein ihtiyacı artar; ancak aşırı besleme doğum güçlüğüne ve metabolik hastalıklara yol açabilir. Doğumdan yaklaşık üç hafta önce geçiş rasyonuna başlanmalı, mineral ve vitamin desteği verilmelidir. Doğum yaklaştığında hayvan temiz ve rahat bir bölmede gözlem altında tutulmalıdır.",
    "Aşı takvimine uyulması hastalıkların önlenmesinin temelidir. Şap, brusella, şarbon ve yanıkara gibi hastalıklara karşı düzenli aşılama yapılmalıdır. Aşı zamanları hayvanın yaşına ve bölgedeki risklere göre veteriner hekim tarafından belirlenir. Aşı sonrası hayvan birkaç gün gözlem altında tutulmalı ve aşı kayıtları düzenli olarak tutulmalıdır.",
    "Süt sağımı hijyeni mastitis (meme iltihabı) riskini azaltmanın en önemli yoludur. Sağımdan önce meme temizlenip kurulanmalı, sağım ekipmanları her kullanımdan sonra dezenfekte edilmelidir. İlk sütte pıhtı, kan veya renk değişikliği görülürse mastitis şüphesiyle veteriner hekime danışılmalıdır. Sağım sonrası memelerin temiz ortamda tutulması enfeksiyon riskini düşürür.",
    "İshal, özellikle buzağılarda tehlikelidir ve hızlı sıvı kaybına yol açar. Nedenleri arasında bakteri, virüs veya parazit enfeksiyonları, ani yem değişikliği, kirli su ve hijyen eksikliği bulunur. Hayvana bol temiz su ve gerekirse elektrolit verilmelidir. İshal bir günden uzun sürerse, kanlıysa veya hayvan halsizse acilen veteriner hekime başvurulmalıdır.",
    "Yeterli ve temiz su, hayvan sağlığı ve süt verimi için kritiktir. Bir süt ineği günde yaklaşık 60-100 litre su içebilir; su kısıtlandığında süt verimi hızla düşer. Su kaynağı temiz, kolay erişilebilir ve sıcak havalarda serin tutulmalıdır. Kirli veya yetersiz su iştahsızlık ve hastalıklara zemin hazırlar.",
    "İç ve dış parazitler hayvanlarda zayıflama, kıl dökülmesi, kaşıntı, kansızlık ve verim düşüklüğüne yol açar. İç parazitler için ilaçlama, dış parazitler için uygun uygulamalar düzenli olarak yapılmalıdır. Parazit kontrol programı, mevsime ve bölge koşullarına göre veteriner hekim önerisiyle planlanmalıdır.",
    "Doğum sırasında hayvan sakin, temiz ve gözlem altında tutulmalıdır. Normal doğum genellikle birkaç saat içinde tamamlanır. Doğum uzarsa, buzağının duruşu ters ise veya hayvan aşırı zorlanıyorsa vakit kaybetmeden veteriner hekim çağrılmalıdır. Doğum sonrası hem ana hem yavru yakından izlenmelidir.",
    "Yeni doğan buzağının ilk saatlerde ağız sütü (kolostrum) alması hayati önemdedir. Kolostrum buzağıya bağışıklık kazandırır ve hastalıklara karşı korur. İlk iki saat içinde, doğum ağırlığının yaklaşık yüzde onu kadar kolostrum verilmelidir. Geciken veya yetersiz kolostrum, buzağının hastalanma ve ölüm riskini belirgin artırır.",
    "Sıcaklık stresi, özellikle yaz aylarında süt ineklerinde verim düşüşüne, iştahsızlığa ve solunum hızlanmasına yol açar. Hayvanlara gölgelik, iyi havalandırma ve bol serin su sağlanmalıdır. Sıcak saatlerde ağır yem yerine daha hafif ve sindirilebilir beslenme tercih edilmelidir. Aşırı sıcak hayvan sağlığı için ciddi bir risktir.",
    "Hayvanların sağlıklı olması ve verimli çalışması için dengeli bir rasyon şarttır. Rasyon; enerji, protein, lif, mineral ve vitaminleri hayvanın ihtiyacına göre içermelidir. Kaba yem (ot, silaj) ve kesif yem (tahıl karması) dengesi önemlidir. Dengesiz besleme, verim düşüklüğüne ve sindirim sorunlarına yol açar.",
    "Şişkinlik (timpani), işkembede aşırı gaz birikmesiyle oluşan, hızlı gelişebilen ciddi bir durumdur. Genellikle aşırı taze veya yaş yonca gibi baklagillerin fazla tüketilmesiyle görülür. Hayvanın sol böğrü belirgin şişer, huzursuzluk ve solunum güçlüğü olur. Şişkinlik acil bir durumdur; derhal veteriner hekime başvurulmalıdır.",
    "Düzenli tırnak bakımı topallığı ve ayak hastalıklarını önlemenin temelidir. Tırnaklar aşırı uzadığında hayvanın duruşu bozulur ve yürüme güçleşir. Ahır zemini kuru, temiz ve kaymayı önleyecek şekilde olmalıdır. Yılda birkaç kez tırnak kesimi ve kontrolü önerilir; belirgin aksama varsa veteriner hekime danışılmalıdır.",
    "Kızgınlık (östrus) belirtilerinin doğru takibi, başarılı tohumlama için gereklidir. Belirtiler arasında huzursuzluk, diğer hayvanlara atlama veya atlanmaya izin verme, iştah değişikliği ve akıntı bulunur. Kızgınlık genellikle belirli aralıklarla tekrarlar. Doğru zamanda tohumlama için kızgınlık günü kayıt altına alınmalıdır.",
    "Buzağılar bağışıklıkları zayıf olduğu için temiz, kuru ve rüzgârdan korunaklı bir barınakta tutulmalıdır. Islak ve kirli zemin, ishal ve solunum hastalıklarına davetiye çıkarır. Barınak düzenli temizlenmeli, altlık kuru tutulmalı ve yeterli temiz hava sağlanmalıdır. Hasta buzağılar sağlıklı olanlardan ayrılmalıdır.",
    "Solunum yolu hastalıkları, özellikle genç hayvanlarda öksürük, burun akıntısı, hızlı solunum ve ateşle kendini gösterir. Nedenleri arasında soğuk, nemli ve kötü havalandırılan barınaklar, ani sıcaklık değişimleri ve enfeksiyonlar bulunur. Erken fark edilirse tedavi başarılıdır; belirtiler görülürse veteriner hekime başvurulmalıdır.",
    "Mineral ve vitamin eksiklikleri; iştahsızlık, zayıflama, tüy ve kıl bozuklukları, üreme sorunları ve verim düşüklüğüne yol açabilir. Özellikle kalsiyum, fosfor, selenyum ve A, D, E vitaminleri önemlidir. Dengeli rasyon ve gerektiğinde mineral takviyesiyle önlenebilir. Takviye programı veteriner hekim önerisiyle belirlenmelidir.",
    "Süt humması (doğum felci), genellikle doğumdan hemen sonra kandaki kalsiyumun ani düşmesiyle görülür. Hayvan halsizleşir, ayağa kalkamaz ve titreme görülebilir. Özellikle yüksek verimli ve yaşlı ineklerde risk daha yüksektir. Bu acil bir durumdur; derhal veteriner hekime başvurulmalıdır.",
    "Yemdeki ani değişiklikler işkembe dengesini bozarak sindirim sorunlarına, iştahsızlığa ve verim düşüklüğüne yol açar. Yeni bir yeme geçiş birkaç gün içinde kademeli yapılmalıdır. İşkembe sağlığı için yeterli kaba yem (lif) verilmesi önemlidir. Ani ve aşırı kesif yem, asidoz gibi sorunlara neden olabilir.",
    "Hayvanların günlük gözlemi, sorunların erken fark edilmesini sağlar. İştah, hareketlilik, dışkı kıvamı, süt verimi ve davranıştaki değişimler önemli göstergelerdir. Kulakların düşük olması, sürüden ayrı durma, iştahsızlık veya verim düşüşü dikkat edilmesi gereken işaretlerdir. Şüpheli durumlarda veteriner hekime danışılmalıdır.",
    "Yem ve su kaplarının düzenli temizliği hastalıkların yayılmasını önler. Kirli kaplarda bakteri ve küf üreyebilir; bu da iştahsızlık ve sindirim sorunlarına yol açar. Su kapları her gün, yemlikler düzenli aralıklarla temizlenmelidir. Küflenmiş veya bozulmuş yem kesinlikle hayvana verilmemelidir.",
    "Sürüye yeni katılan hayvanlar, hastalık taşıma riskine karşı bir süre karantinada (ayrı) tutulmalıdır. Bu sürede hayvan gözlemlenmeli, gerekli aşı ve parazit kontrolleri yapılmalıdır. Karantina, bulaşıcı hastalıkların tüm sürüye yayılmasını önleyen önemli bir koruyucu tedbirdir. Şüpheli belirtilerde veteriner hekime danışılmalıdır."
]

bilgi_embed = embed_model.encode(bilgi_tabani, convert_to_tensor=True)
print("Bilgi tabanı:", len(bilgi_tabani), "konu")

Bilgi tabanı: 24 konu


In [4]:
from transformers import pipeline
import torch
llm = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct",
               torch_dtype=torch.float16, device_map="auto")
print("Model yüklendi.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model yüklendi.


Açıklama: Cevap üreten dil modeli (Qwen 3B).

In [5]:
def konu_uygun_mu(soru):
    kontrol = [
        {"role": "system", "content":
         "Görevin: sorunun hayvan/çiftlik/veteriner konusuyla ilgili olup olmadığına "
         "karar vermek. İlgiliyse 'EVET', değilse 'HAYIR'.\n"
         "'inek neden süt vermiyor' -> EVET\n'buzağıda ishal var' -> EVET\n"
         "'sıcakta hayvana ne yapmalıyım' -> EVET\n'hava durumu nasıl' -> HAYIR\n"
         "'telefonum bozuldu' -> HAYIR\nSadece EVET veya HAYIR yaz."},
        {"role": "user", "content": soru}
    ]
    cevap = llm(kontrol, max_new_tokens=5, do_sample=False)
    return "EVET" in cevap[0]["generated_text"][-1]["content"].upper()

asistan fonksiyonu (few-shot'lu, 08'deki akıllı sürüm)

In [6]:
hayvan_veritabani = {
    "TR-001": {"ad": "Sarıkız", "yas": 4, "agirlik": 520, "gunluk_verim": 14.5,
               "anomali": False, "laktasyon": "orta", "not": "-"},
    "TR-002": {"ad": "Benekli", "yas": 6, "agirlik": 410, "gunluk_verim": 6.2,
               "anomali": True, "laktasyon": "geç", "not": "kilo düşük, verim az"},
    "TR-003": {"ad": "Karakaş", "yas": 3, "agirlik": 495, "gunluk_verim": 17.0,
               "anomali": False, "laktasyon": "pik", "not": "-"},
}
print("Kayıtlı hayvan:", len(hayvan_veritabani))

Kayıtlı hayvan: 3


Açıklama: Her hayvanın gerçek verisi (yaş, ağırlık, günlük verim, anomali durumu, laktasyon dönemi). Gerçek uygulamada bu veritabanından gelecek; TR-002 kasıtlı olarak "sorunlu" (düşük kilo, düşük verim, anomali) ki kişiselleştirmeyi test edelim.

In [8]:
def kisisel_asistan(kupe_no, soru):
    # Hayvan var mı kontrol
    if kupe_no not in hayvan_veritabani:
        return f"'{kupe_no}' numaralı hayvan kayıtlı değil."

    h = hayvan_veritabani[kupe_no]

    # Konu kontrolü
    if not konu_uygun_mu(soru):
        return "Ben bir veteriner asistanıyım, yalnızca hayvan sağlığı konularında yardımcı olabilirim."

    # RAG: bilgi tabanından en yakın 2 metin
    soru_embed = embed_model.encode(soru, convert_to_tensor=True)
    benzerlikler = util.cos_sim(soru_embed, bilgi_embed)[0]
    en_iyi = benzerlikler.argsort(descending=True)[:2]
    genel_bilgi = "\n\n".join([bilgi_tabani[i] for i in en_iyi])

    # Hayvanın kendi verisini metne çevir
    hayvan_bilgisi = (
        f"Küpe: {kupe_no}, Ad: {h['ad']}, Yaş: {h['yas']}, Ağırlık: {h['agirlik']} kg, "
        f"Günlük süt verimi: {h['gunluk_verim']} L, Laktasyon: {h['laktasyon']}, "
        f"Anomali durumu: {'VAR (sürüden sapıyor)' if h['anomali'] else 'yok'}. "
        f"Not: {h['not']}"
    )

    # LLM'e hem genel bilgiyi hem hayvanın verisini ver
    mesaj = [
        {"role": "system", "content":
         "Sen bir veteriner asistanısın. Sana verilen GENEL BİLGİ ve HAYVANIN "
         "KENDİ VERİSİNİ birlikte kullanarak, bu hayvana ÖZEL bir cevap ver. "
         "Hayvanın verisindeki dikkat çekici noktaları (düşük ağırlık, düşük verim, "
         "anomali gibi) yorumla. Teşhis koyma, sonunda veteriner hekime yönlendir."},
        {"role": "user", "content":
         f"GENEL BİLGİ:\n{genel_bilgi}\n\nHAYVANIN VERİSİ:\n{hayvan_bilgisi}\n\nSORU: {soru}"}
    ]
    cevap = llm(mesaj, max_new_tokens=250, do_sample=False)
    return cevap[0]["generated_text"][-1]["content"]


# Test: sorunlu hayvan (TR-002, düşük verim + anomali)
print("🐮 TR-002 (Benekli) için soru: 'Bu ineğim neden az süt veriyor?'\n")
print(kisisel_asistan("TR-002", "bu ineğim neden az süt veriyor?"))

[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🐮 TR-002 (Benekli) için soru: 'Bu ineğim neden az süt veriyor?'



[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bu hayvanın durumu oldukça dikkat çekicidir. Genel bilgiler ve hayvanın verisi göz önüne alındığında, Benekli hayvanının az süt verimi nedeni genellikle su kaynaklarının yetersizliği ve/veya kirliliğidir. 

Hayvanın ağırlığı 410 kg ve günlük süt verimi 6.2 litreyken, genel olarak hayvanlar günde yaklaşık 60-100 litre su içebilir. Bu da su kaynaklarının yetersizliği anlamına gelir. Ayrıca, hayvanın laktasyonu "geç" durumunda ve kilosu düşük durumda, bu da su kaynaklarının yetersizliği ve kirliliğinin olası etkilerini göstermektedir.

Anomali durumu "sürüden sapıyor" olarak belirtilmiş, bu da su kaynaklarının yetersizliği veya kirliliğinin hayvanın fiziksel ve beslenme ihtiyaçlarını karşılamadığı anlamına gelmektedir. Bu durum, hay


kisisel_asistan iki şeyi birleştiriyor — (1) bilgi tabanından genel bilgi (RAG), (2) o hayvanın kendi verisi (ağırlık, verim, anomali). İkisini birden LLM'e verip, "bu hayvana özel" cevap ürettiriyoruz. TR-002 sorunlu bir hayvan (düşük kilo, düşük verim, anomali VAR), o yüzden asistan sadece "süt düşüşü şöyle olur" demeyecek, "senin bu ineğinin ağırlığı düşük ve anomali işaretli, bu yüzden..." diye kişiye özel konuşacak.

In [9]:
print("🐮 TR-003 (Karakaş) — sağlıklı, yüksek verimli:\n")
print(kisisel_asistan("TR-003", "bu ineğimin durumu nasıl, dikkat etmem gereken bir şey var mı?"))

[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🐮 TR-003 (Karakaş) — sağlıklı, yüksek verimli:



[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Karakaş inekinizin verimliliği ve genel sağlık durumunuz açısından bazı dikkat çeken noktalar bulunmaktadır:

1. **Ağırlık**: Hayvanınızın ağırlığı 495 kg olarak verilmiştir. Bu, normal bir ağırlık aralığında bulunuyor. Ancak, hayvanın ağırlığı sürekli değişebilir ve bu da verimliliğe etkilenebilir. 

2. **Günlük Süt Verimi**: Günlük süt verimi 17.0 L olarak verilmiştir. Bu, genellikle normal bir verimlilik seviyesidir. Ancak, verimliliklerin sürekli monitoringu önemlidir ve herhangi bir düşüş durumunda dikkat edilmesi gerekmektedir.

3. **Anomali Durumu**: Verilen bilgide "Anomali durumu: yok" olarak belirtilmiştir. Bu, verimliliğinizin ve genel sağlığınızın tamamen normal olduğunu göstermektedir.

4. **Düş


## Kapanış — Kişiselleştirilmiş Asistan (Yol Haritası 1. Adım)

Bu notebook'ta RAG asistanını, belirli bir hayvanın gerçek verileriyle birleştirdik.

**Yapılanlar**
- Her hayvanın verisini (yaş, ağırlık, günlük verim, anomali durumu, laktasyon)
  tutan bir yapı oluşturuldu
- Asistan, cevap üretirken hem genel bilgi tabanını hem de o hayvanın kendi
  verisini birlikte kullanacak şekilde geliştirildi
- Sorunlu (TR-002: düşük kilo + anomali) ve sağlıklı (TR-003) hayvanlarla test edildi

**Sonuç**
Asistan artık aynı soruya farklı hayvanlar için farklı cevap veriyor: sorunlu
hayvanda düşük ağırlık ve anomaliyi vurgulayıp dikkat çekiyor, sağlıklı hayvanda
"durumu normal" diyor. Genel bilgi botundan, sürüyü tanıyan kişisel bir danışmana
dönüştü.

**Yol haritası:** 1. adım (Kişiselleştirme) ✅ tamamlandı.